In [5]:
import pandas as pd
import json
import base64
import time
from pathlib import Path
from datetime import datetime
from pdf2image import convert_from_path
import mimetypes
from typing import Optional, List

from anthropic import Anthropic


# ==========================
# CONFIGURATION (COMPANY GATEWAY)
# ==========================
base_url = "https://anthropic.prod.ai-gateway.quantumblack.com/7a4f3d63-b5db-4d5b-8076-8f114d1f14f7"
access_token = "eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJhZXNKN2kxNGNidnVuTU40MTJrOU5yZ2ROeENhTlJudTNPbC1TU08ycFlJIn0.eyJleHAiOjE3NjUyNTY1MjksImlhdCI6MTc2NTI1NDcyOSwiYXV0aF90aW1lIjoxNzY1MjU0NzI4LCJqdGkiOiI4ZTU0ZDgwOC05MmU5LTQ3YTYtYWUyZi0zNGQzYjk5MWZmYmIiLCJpc3MiOiJodHRwczovL2F1dGgubWNraW5zZXkuaWQvYXV0aC9yZWFsbXMvciIsImF1ZCI6ImJjZDIzNzI4LTNkMjctNDQ3Yy1hMGE5LWVhY2FmMzkzYTZmNSIsInN1YiI6IjI0NDRiYzZjLTAwMzctNGIyZS1hYzI3LWZjNTlhNTkxNTM2NiIsInR5cCI6IklEIiwiYXpwIjoiYmNkMjM3MjgtM2QyNy00NDdjLWEwYTktZWFjYWYzOTNhNmY1Iiwic2Vzc2lvbl9zdGF0ZSI6IjRlOTFiMWVhLTY4M2UtNDI1My05Zjg3LTNlYzhlMzAyMmQ1MiIsImF0X2hhc2giOiJ0QzF4NWRuRnhMZWw1X3AzaHFfRlZBIiwibmFtZSI6IlVnYW5kaGFyIFZhZGRpIiwiZ2l2ZW5fbmFtZSI6IlVnYW5kaGFyIiwiZmFtaWx5X25hbWUiOiJWYWRkaSIsInByZWZlcnJlZF91c2VybmFtZSI6IjE1ZDhiYmNkMmMzNTNmYWUiLCJlbWFpbCI6IlVnYW5kaGFyX1ZhZGRpQG1ja2luc2V5LmNvbSIsImFjciI6IjEiLCJzaWQiOiI0ZTkxYjFlYS02ODNlLTQyNTMtOWY4Ny0zZWM4ZTMwMjJkNTIiLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwiZm1ubyI6IjM0NzI3NiIsImdyb3VwcyI6WyI3YTRmM2Q2My1iNWRiLTRkNWItODA3Ni04ZjExNGQxZjE0ZjciLCJBbGwgRmlybSBVc2VycyJdfQ.df9PWvr-r_hg_tZzleq6OZ6a8cSWdCJ403yb2EPQ9dtbNVI8amb5wgkV5Vl29rDNc9VBdZfepi-K4mx5S44SADHVxEcnQnK8gfik-t4FA-ZfjlDxgNdcPQhPSEjCw7tBALCcYPKGcDYWBT8YECuPNExPhuvCX5IY5IQ7zjrGMZWzoq-u2XZcXzhp1HK2KEoEXBuFz3i3tsqN280syj2dpXofiAJHqDLeuKxWp2YMBJN6_mhtEmo5uBeff7OWEKe7F7oFaiVFFNLIl3Ae5SdFIT98ItHdj_MooHB1XK5flk1vXBKtl2EhplM93Ci7WWGZa8QGgUPoAoVMDpRqNM6BkQ"

client = Anthropic(
    base_url=base_url,
    api_key=access_token
)

MODEL_NAME = "claude-sonnet-4-20250514"


# ==========================
# PROMPT (NO ALT UNITS)
# ==========================
def get_dimension_prompt():
    return """
    You are an Engineering Dimension Extraction AI specialized in reading
    dimensioned mechanical drawings. You must ONLY extract values that are
    explicitly written on the drawing.

    SECTION 1 – TITLE BLOCK METADATA
    - title: The main drawing title from the title block.
    - drawing_number: The drawing / part / print number from the title block.

    SECTION 2 – DIMENSION EXTRACTION RULES

    Extract the following parameters from the drawing:

    A) SINGLE UNIT PARAMETERS:
       - length
       - inner_diameter (ID)
       - outer_diameter (OD)
       - material (MATL)

    B) DUAL PARAMETERS (BUT YOU ONLY OUTPUT ONE UNIT):
       - surface_area (S/A)
       - weight (WT)

    IMPORTANT UNIT RESTRICTIONS:
    - For surface_area: ONLY output if explicitly given in square inches
      (examples: "in²", "in^2", "sq in", "square inches").
    - For weight: ONLY output if explicitly given in pounds
      (examples: "lb", "lbs", "pound", "pounds").
    - If the drawing shows surface area or weight in any OTHER unit
      (e.g., cm², m², kg, g, N), set the value and unit to null.
    - DO NOT perform any unit conversions.

    SINGLE UNIT PARAMETERS (length, inner_diameter, outer_diameter, material):
    - Extract ONLY the primary unit shown on the drawing.
    - Format: {parameter}_value and {parameter}_unit.
    - Even if a second unit is shown in parentheses, IGNORE it.

    SURFACE AREA:
    - Look for labels like: S/A, Surface Area, SURF AREA.
    - If square inches are explicitly shown, extract:
        surface_area_value, surface_area_unit
      (where unit is written as "in^2").
    - If square inches are NOT explicitly shown or are not readable,
      set surface_area_value = null and surface_area_unit = null.

    WEIGHT:
    - Look for labels like: WT, Weight, WEIGHT, Wt.
    - If pounds are explicitly shown, extract:
        weight_value, weight_unit
      (where unit is written as "lb").
    - If pounds are NOT explicitly shown or are not readable,
      set weight_value = null and weight_unit = null.

    MATERIAL:
    - Look for labels like: MATL, MATERIAL, MAT'L.
    - Extract exact text as shown.
    - If not explicitly labeled, set material = null.

    NULL HANDLING:
    - If a value is not explicitly labeled, set value = null and unit = null.
    - Missing is better than wrong.
    - When in doubt, use null.

    OUTPUT FORMAT (STRICT JSON, NO MARKDOWN):

    {
      "title": "SPACER RING",
      "drawing_number": "DRW-12345-A",

      "length_value": 100.0,
      "length_unit": "mm",

      "inner_diameter_value": 20.0,
      "inner_diameter_unit": "mm",

      "outer_diameter_value": 30.0,
      "outer_diameter_unit": "mm",

      "material": "Aluminum 6061",

      "surface_area_value": 23.6,
      "surface_area_unit": "in^2",

      "weight_value": 0.52,
      "weight_unit": "lb"
    }

    FIELD DEFINITIONS:
    - All *_value fields: numeric (float) or null
    - All *_unit fields: string or null
    - material: string or null
    - title: string or null
    - drawing_number: string or null
    """


# ==========================
# NORMALIZATION HELPERS
# ==========================
def _normalize_surface_area_unit(unit: Optional[str]) -> Optional[str]:
    """Return 'in^2' if the unit is any variant of square inches, else None."""
    if not unit:
        return None
    u = unit.strip().lower().replace(" ", "")
    sq_in_variants = {
        "in^2",
        "in²",
        "sqin",
        "sq.in",
        "squareinches",
        "squareinch",
        "in2",
    }
    if u in sq_in_variants:
        return "in^2"
    return None


def _normalize_weight_unit(unit: Optional[str]) -> Optional[str]:
    """Return 'lb' if the unit is any variant of pounds, else None."""
    if not unit:
        return None
    u = unit.strip().lower()
    if u in {"lb", "lbs", "pound", "pounds"}:
        return "lb"
    return None


def cleanup_extracted_data(data: dict) -> dict:
    """
    Enforce:
    - No alt fields at all.
    - surface_area only if in square inches.
    - weight only if in pounds.
    """
    keys_defaults = {
        "title": None,
        "drawing_number": None,
        "material": None,
        "length_value": None,
        "length_unit": None,
        "inner_diameter_value": None,
        "inner_diameter_unit": None,
        "outer_diameter_value": None,
        "outer_diameter_unit": None,
        "surface_area_value": None,
        "surface_area_unit": None,
        "weight_value": None,
        "weight_unit": None,
    }
    for k, v in keys_defaults.items():
        data.setdefault(k, v)

    # Drop any *_alt fields if model returned them
    for key in list(data.keys()):
        if key.endswith("_value_alt") or key.endswith("_unit_alt"):
            data.pop(key, None)

    # surface area normalization
    sa_unit_norm = _normalize_surface_area_unit(data.get("surface_area_unit"))
    if sa_unit_norm is None or data.get("surface_area_value") is None:
        data["surface_area_value"] = None
        data["surface_area_unit"] = None
    else:
        data["surface_area_unit"] = sa_unit_norm

    # weight normalization
    wt_unit_norm = _normalize_weight_unit(data.get("weight_unit"))
    if wt_unit_norm is None or data.get("weight_value") is None:
        data["weight_value"] = None
        data["weight_unit"] = None
    else:
        data["weight_unit"] = wt_unit_norm

    return data


# ==========================
# LOW-LEVEL CALL (image bytes → dict)
# ==========================
def _call_model_on_image_bytes(image_bytes: bytes, mime_type: str) -> dict:
    # Convert image to base64
    image_data = base64.standard_b64encode(image_bytes).decode("utf-8")
    
    # Create message with image
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=4096,
        temperature=0.1,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": mime_type,
                            "data": image_data,
                        },
                    },
                    {
                        "type": "text",
                        "text": get_dimension_prompt()
                    }
                ],
            }
        ],
    )

    raw = response.content[0].text

    if "```json" in raw:
        raw = raw.split("```json", 1)[1].split("```", 1)[0]
    elif "```" in raw:
        raw = raw.split("```", 1)[1].split("```", 1)[0]

    try:
        data = json.loads(raw.strip())
    except json.JSONDecodeError:
        data = {}

    if not isinstance(data, dict):
        data = {}

    return data


# ==========================
# IMAGE ANALYSIS (ONE PAGE)
# ==========================
def analyze_image(image_path: str) -> dict:
    with open(image_path, "rb") as f:
        image_bytes = f.read()

    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/png"

    try:
        data = _call_model_on_image_bytes(image_bytes, mime_type)
    except Exception:
        data = {}

    data = cleanup_extracted_data(data)
    return data


# ==========================
# PDF → IMAGES
# ==========================
def convert_file_to_images(file_path: str, dpi: int = 300):
    """
    Convert PDF to images.
    If image already, just return [file_path].
    Returns (page_images, is_pdf)
    """
    ext = Path(file_path).suffix.lower()
    if ext == ".pdf":
        pages = convert_from_path(file_path, dpi=dpi)
        if not pages:
            return [], True
        ts = int(time.time())
        imgs = []
        for i, p in enumerate(pages, start=1):
            img = f"temp_dim_{i}_{ts}.png"
            p.save(img, "PNG")
            imgs.append(img)
        return imgs, True
    else:
        return [file_path], False


# ==========================
# BUILD DATAFRAME
# ==========================
def build_output_dataframe(rows: List[dict]) -> pd.DataFrame:
    column_order = [
        "Page",
        "title",
        "drawing_number",
        "length_value",
        "length_unit",
        "inner_diameter_value",
        "inner_diameter_unit",
        "outer_diameter_value",
        "outer_diameter_unit",
        "material",
        "surface_area_value",
        "surface_area_unit",
        "weight_value",
        "weight_unit",
    ]
    df = pd.DataFrame(rows)
    for col in column_order:
        if col not in df.columns:
            df[col] = None
    return df[column_order]


# ==========================
# MAIN PROCESSOR
# ==========================
def process_file_enhanced(file_path: str):
    """
    - PDF → per-page images (300 DPI) or image directly
    - Per page: call Claude, clean data
    - Export to single Excel with timestamp
    - Console: only error or final 'Done: <file>'
    """
    path_obj = Path(file_path)
    if not path_obj.exists():
        print("ERROR: File not found")
        return

    page_images, is_pdf = convert_file_to_images(file_path, dpi=300)
    if not page_images:
        print("ERROR: No images to process")
        return

    rows = []
    try:
        for i, img in enumerate(page_images, start=1):
            result = analyze_image(img)
            result["Page"] = i
            rows.append(result)

        df = build_output_dataframe(rows)

        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_name = f"{path_obj.stem}_Dimensions_Extracted_{ts}.xlsx"
        df.to_excel(out_name, index=False)
        print(f"Done: {out_name}")

    finally:
        if is_pdf:
            for img in page_images:
                try:
                    Path(img).unlink(missing_ok=True)
                except Exception:
                    pass


# ==========================
# ENTRY POINT
# ==========================
if __name__ == "__main__":
    # Example:
    # process_file_enhanced("your_drawing.pdf")
    # process_file_enhanced("part_drawing.png")
    print("Module loaded. Call process_file_enhanced('your_drawing.pdf').")

Module loaded. Call process_file_enhanced('your_drawing.pdf').


In [6]:
file_path = "Master.png"
#file_path = "https://www.shutterstock.com/image-vector/sketch-bushing-vector-eps10-260nw-111717389.jpg"
process_file_enhanced(file_path)

Done: Master_Dimensions_Extracted_20251209_102258.xlsx
